In [1]:
import joblib

X_train = joblib.load(
    "X_train_random.pkl"
)

X_test = joblib.load(
    "X_test_random.pkl"
)

y_train = joblib.load(
    "y_train_random.pkl"
)

y_test = joblib.load(
    "y_test_random.pkl"
)

In [2]:
import numpy as np
import pandas as pd
import optuna

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import RobustScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_regression
)

from sklearn.model_selection import cross_val_score

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor

In [3]:
preprocessor_robust = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="mean")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.01)
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=mutual_info_regression,
            k=1000
        )
    )

])

In [8]:
def objective(trial):

    model = XGBRegressor(

        n_estimators=trial.suggest_int(
            "n_estimators",
            200,
            1200
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            12
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        min_child_weight=trial.suggest_int(
            "min_child_weight",
            1,
            20
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        gamma=trial.suggest_float(
            "gamma",
            0,
            5
        ),

        reg_alpha=trial.suggest_float(
            "reg_alpha",
            0,
            10
        ),

        reg_lambda=trial.suggest_float(
            "reg_lambda",
            0,
            10
        ),

        objective="reg:squarederror",

        random_state=42,

        n_jobs=1
    )

    pipeline = Pipeline([

        (
            "preprocessor_robust",
            preprocessor_robust
        ),

        (
            "model",
            model
        )

    ])

    scores = cross_val_score(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring="r2",

        n_jobs=-1

    )

    return scores.mean()

In [9]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

[I 2026-06-02 21:05:48,220] A new study created in memory with name: no-name-effa7d93-efb1-4cbb-9c69-3756c7f8ed97


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-06-02 21:09:51,196] Trial 0 finished with value: 0.786737845114296 and parameters: {'n_estimators': 555, 'max_depth': 4, 'learning_rate': 0.07428933759509777, 'min_child_weight': 10, 'subsample': 0.8474731505547697, 'colsample_bytree': 0.9282975493682838, 'gamma': 0.6134645172464337, 'reg_alpha': 8.129000590339109, 'reg_lambda': 4.544070573314155}. Best is trial 0 with value: 0.786737845114296.
[I 2026-06-02 21:14:13,978] Trial 1 finished with value: 0.7863330728561687 and parameters: {'n_estimators': 861, 'max_depth': 5, 'learning_rate': 0.038548686260174145, 'min_child_weight': 3, 'subsample': 0.924763819686494, 'colsample_bytree': 0.7363567525103394, 'gamma': 2.981468373013674, 'reg_alpha': 1.972165082116275, 'reg_lambda': 3.2182976685612097}. Best is trial 0 with value: 0.786737845114296.
[I 2026-06-02 21:18:18,441] Trial 2 finished with value: 0.7757762385428786 and parameters: {'n_estimators': 300, 'max_depth': 11, 'learning_rate': 0.16945556443700754, 'min_child_weight':

In [10]:
print("Best CV R2:")
print(study.best_value)



print("Best Parameters:")
print(study.best_params)

Best CV R2:
0.7933474224915333
Best Parameters:
{'n_estimators': 1186, 'max_depth': 11, 'learning_rate': 0.010447399837617896, 'min_child_weight': 20, 'subsample': 0.7357675729803006, 'colsample_bytree': 0.6150588031463851, 'gamma': 0.2669367398873921, 'reg_alpha': 4.47372043956587, 'reg_lambda': 9.851978482686171}


In [11]:


best_xgb_pipeline = Pipeline([

    (
        "preprocessor_robust",
        preprocessor_robust
    ),

    (
        "model",
        XGBRegressor(

            **study.best_params,

            objective="reg:squarederror",

            random_state=42,

            n_jobs=-1
        )
    )

])


# TRAIN MODEL

best_xgb_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor_robust', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('imputer', ...), ('variance_filter', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'mean'
,fill_value,None


In [12]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

import numpy as np


# TRAIN PREDICTIONS

y_train_pred = best_xgb_pipeline.predict(
    X_train
)


# TEST PREDICTIONS

y_test_pred = best_xgb_pipeline.predict(
    X_test
)


# TRAIN METRICS

train_r2 = r2_score(
    y_train,
    y_train_pred
)

train_rmse = np.sqrt(
    mean_squared_error(
        y_train,
        y_train_pred
    )
)

train_mae = mean_absolute_error(
    y_train,
    y_train_pred
)


# TEST METRICS

test_r2 = r2_score(
    y_test,
    y_test_pred
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

test_mae = mean_absolute_error(
    y_test,
    y_test_pred
)


# RESULTS

print("\n TUNED XGBOOST ")

print("\nTrain R2 :", train_r2)
print("Train RMSE :", train_rmse)
print("Train MAE :", train_mae)

print("\nTest R2 :", test_r2)
print("Test RMSE :", test_rmse)
print("Test MAE :", test_mae)


 TUNED XGBOOST 

Train R2 : 0.9428621282241303
Train RMSE : 0.5683471416530658
Train MAE : 0.3228537425438731

Test R2 : 0.8184108904317446
Test RMSE : 0.9922199610750365
Test MAE : 0.6446366332963599


In [14]:
print(study.best_value)

print(train_r2)
print(test_r2)

print(test_rmse)
print(test_mae)

0.7933474224915333
0.9428621282241303
0.8184108904317446
0.9922199610750365
0.6446366332963599


In [15]:
print("Best Parameters:")
print(study.best_params)

Best Parameters:
{'n_estimators': 1186, 'max_depth': 11, 'learning_rate': 0.010447399837617896, 'min_child_weight': 20, 'subsample': 0.7357675729803006, 'colsample_bytree': 0.6150588031463851, 'gamma': 0.2669367398873921, 'reg_alpha': 4.47372043956587, 'reg_lambda': 9.851978482686171}


In [ ]:
"""import joblib

joblib.dump(
    tuned_xgboost_pipeline,
    "xgboost_optuna_robust_pipeline.pkl"
)